# Coupled Thermal Dimer: Model Construction

This notebook constructs and validates the coupled two-level-system
model used in Section 6.

The goals are:

1. Construct the two-TLS Hilbert-space operators.
2. Construct the Hamiltonian and dipole operator.
3. Construct the local thermal Lindblad operators.
4. Build the Liouvillian superoperator using column-stacking vectorization.
5. Build the light-matter interaction superoperator.
6. Perform basic consistency checks before spectroscopy.

In [1]:
import numpy as np

from numpy.linalg import eig, eigvals, norm
from scipy.linalg import expm

## 1. Baseline parameters

In [2]:
# Units: hbar = 1

hbar = 1.0

# Two-level-system transition frequencies
omega_a = 1.0
omega_b = 1.2

# Coherent coupling
J = 0.10

# Local relaxation rates
gamma_a = 0.05
gamma_b = 0.05

# Mean thermal occupations
nbar_a = 0.05
nbar_b = 0.05

# Transition dipole amplitudes
mu_a = 1.0
mu_b = 1.0

## 2. Single-TLS operators and composite basis

We use the single-TLS basis

$$
\{|g\rangle, |e\rangle\},
$$

with

$$
|g\rangle =
\begin{pmatrix}
1\\
0
\end{pmatrix},
\qquad
|e\rangle =
\begin{pmatrix}
0\\
1
\end{pmatrix}.
$$

The composite basis is ordered as

$$
\{|gg\rangle, |eg\rangle, |ge\rangle, |ee\rangle\}.
$$

To obtain this ordering with NumPy Kronecker products, subsystem $b$
is taken as the first tensor factor and subsystem $a$ as the second:

$$
|b\rangle\otimes|a\rangle.
$$

Thus,

$$
\sigma_a^\pm = I\otimes\sigma^\pm,
\qquad
\sigma_b^\pm = \sigma^\pm\otimes I.
$$

In [3]:
# Single-TLS basis states
g = np.array([1.0, 0.0], dtype=complex)
e = np.array([0.0, 1.0], dtype=complex)

# Single-TLS identity
I2 = np.eye(2, dtype=complex)

# Raising and lowering operators
sigma_plus = np.outer(e, g.conj())
sigma_minus = np.outer(g, e.conj())

print("sigma_plus =")
print(sigma_plus)

print("\nsigma_minus =")
print(sigma_minus)

sigma_plus =
[[0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j]]

sigma_minus =
[[0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j]]


In [4]:
# Composite basis.
# Tensor-product ordering is |b> ⊗ |a> so that the array ordering is
# |gg>, |eg>, |ge>, |ee>.

gg = np.kron(g, g)
eg = np.kron(g, e)
ge = np.kron(e, g)
ee = np.kron(e, e)

basis = np.column_stack([gg, eg, ge, ee])

print("Composite basis matrix:")
print(basis)

Composite basis matrix:
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]


In [5]:
# Operators acting on subsystem a
sigma_a_plus = np.kron(I2, sigma_plus)
sigma_a_minus = np.kron(I2, sigma_minus)

# Operators acting on subsystem b
sigma_b_plus = np.kron(sigma_plus, I2)
sigma_b_minus = np.kron(sigma_minus, I2)

In [6]:
print("sigma_a^+ |gg> =")
print(sigma_a_plus @ gg)

print("\nExpected |eg> =")
print(eg)

print("\nsigma_b^+ |gg> =")
print(sigma_b_plus @ gg)

print("\nExpected |ge> =")
print(ge)

sigma_a^+ |gg> =
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]

Expected |eg> =
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]

sigma_b^+ |gg> =
[0.+0.j 0.+0.j 1.+0.j 0.+0.j]

Expected |ge> =
[0.+0.j 0.+0.j 1.+0.j 0.+0.j]


In [7]:
print(
    "a excitation correct:",
    np.allclose(sigma_a_plus @ gg, eg)
)

print(
    "b excitation correct:",
    np.allclose(sigma_b_plus @ gg, ge)
)

a excitation correct: True
b excitation correct: True


In [8]:
exchange = (
    sigma_a_plus @ sigma_b_minus
    + sigma_a_minus @ sigma_b_plus
)

print("Exchange acting on |ge>:")
print(exchange @ ge)

print("\nExpected |eg>:")
print(eg)

print("\nExchange acting on |eg>:")
print(exchange @ eg)

print("\nExpected |ge>:")
print(ge)

Exchange acting on |ge>:
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]

Expected |eg>:
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]

Exchange acting on |eg>:
[0.+0.j 0.+0.j 1.+0.j 0.+0.j]

Expected |ge>:
[0.+0.j 0.+0.j 1.+0.j 0.+0.j]


In [9]:
print(
    "|ge> -> |eg>:",
    np.allclose(exchange @ ge, eg)
)

print(
    "|eg> -> |ge>:",
    np.allclose(exchange @ eg, ge)
)

|ge> -> |eg>: True
|eg> -> |ge>: True


## 3. System Hamiltonian

The coupled two-level-system Hamiltonian is

$$
H_S
=
\omega_a \sigma_a^+\sigma_a^-
+
\omega_b \sigma_b^+\sigma_b^-
+
J
\left(
\sigma_a^+\sigma_b^-
+
\sigma_a^-\sigma_b^+
\right),
$$

where we use units with $\hbar=1$.

The operators

$$
n_a=\sigma_a^+\sigma_a^-,
\qquad
n_b=\sigma_b^+\sigma_b^-,
$$

count the excitation on each subsystem.

In the basis

$$
\{|gg\rangle,|eg\rangle,|ge\rangle,|ee\rangle\},
$$

we expect

$$
H_S=
\begin{pmatrix}
0 & 0 & 0 & 0\\
0 & \omega_a & J & 0\\
0 & J & \omega_b & 0\\
0 & 0 & 0 & \omega_a+\omega_b
\end{pmatrix}.
$$

In [10]:
# Excitation-number operators
n_a = sigma_a_plus @ sigma_a_minus
n_b = sigma_b_plus @ sigma_b_minus

print("n_a =")
print(n_a)

print("\nn_b =")
print(n_b)

n_a =
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]

n_b =
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]


In [11]:
H_S = (
    hbar * omega_a * n_a
    + hbar * omega_b * n_b
    + hbar * J * exchange
)

print("H_S =")
print(H_S)

H_S =
[[0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 1. +0.j 0.1+0.j 0. +0.j]
 [0. +0.j 0.1+0.j 1.2+0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 2.2+0.j]]


In [12]:
H_expected = hbar * np.array(
    [
        [0.0,     0.0,     0.0,             0.0],
        [0.0, omega_a,       J,             0.0],
        [0.0,       J, omega_b,             0.0],
        [0.0,     0.0,     0.0, omega_a + omega_b],
    ],
    dtype=complex,
)

print(
    "Hamiltonian matches expected matrix:",
    np.allclose(H_S, H_expected)
)

Hamiltonian matches expected matrix: True


In [13]:
hermiticity_error_H = norm(H_S - H_S.conj().T)

print("Hamiltonian Hermiticity error =", hermiticity_error_H)

Hamiltonian Hermiticity error = 0.0


In [14]:
N = n_a + n_b

print("Total excitation-number operator N =")
print(N)

Total excitation-number operator N =
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 2.+0.j]]


In [15]:
comm_H_N = H_S @ N - N @ H_S

print("[H_S, N] =")
print(comm_H_N)

print(
    "\nExcitation-number conservation error =",
    norm(comm_H_N)
)

[H_S, N] =
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]]

Excitation-number conservation error = 0.0


### System eigenstates and eigenenergies

The coherent coupling mixes the local one-excitation states
$|eg\rangle$ and $|ge\rangle$. We therefore diagonalize $H_S$
and compare its eigenenergies with the analytical result.

In [16]:
energies, eigenstates = np.linalg.eigh(H_S)

print("System eigenenergies:")
for i, energy in enumerate(energies):
    print(f"E_{i} = {energy:.10f}")

System eigenenergies:
E_0 = 0.0000000000
E_1 = 0.9585786438
E_2 = 1.2414213562
E_3 = 2.2000000000


In [17]:
Omega_minus = (
    (omega_a + omega_b) / 2
    - np.sqrt(((omega_a - omega_b) / 2)**2 + J**2)
)

Omega_plus = (
    (omega_a + omega_b) / 2
    + np.sqrt(((omega_a - omega_b) / 2)**2 + J**2)
)

print("Analytical one-excitation frequencies:")
print("Omega_- =", Omega_minus)
print("Omega_+ =", Omega_plus)

Analytical one-excitation frequencies:
Omega_- = 0.9585786437626906
Omega_+ = 1.2414213562373095


In [18]:
expected_energies = np.array(
    [
        0.0,
        Omega_minus,
        Omega_plus,
        omega_a + omega_b,
    ]
)

print(
    "Eigenenergies match analytical result:",
    np.allclose(energies, expected_energies)
)

Eigenenergies match analytical result: True


## 4. Dipole operator and optical transitions

The total dipole operator is

$$
\mu
=
\mu_a(\sigma_a^+ + \sigma_a^-)
+
\mu_b(\sigma_b^+ + \sigma_b^-).
$$

The first term changes the excitation number of subsystem $a$ by one,
while the second does the same for subsystem $b$.

In the product basis

$$
\{|gg\rangle, |eg\rangle, |ge\rangle, |ee\rangle\},
$$

the dipole operator connects adjacent excitation-number manifolds:

$$
N=0 \leftrightarrow N=1
\leftrightarrow N=2.
$$

It does not directly connect $|gg\rangle$ to $|ee\rangle$, because the
dipole operator changes the total excitation number by only one.

In [19]:
# Total dipole operator

mu = (
    mu_a * (sigma_a_plus + sigma_a_minus)
    + mu_b * (sigma_b_plus + sigma_b_minus)
)

print("Dipole operator mu =")
print(mu)

Dipole operator mu =
[[0.+0.j 1.+0.j 1.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 1.+0.j 1.+0.j 0.+0.j]]


In [20]:
mu_expected = np.array(
    [
        [0.0,  mu_a,  mu_b, 0.0],
        [mu_a, 0.0,   0.0,  mu_b],
        [mu_b, 0.0,   0.0,  mu_a],
        [0.0,  mu_b,  mu_a, 0.0],
    ],
    dtype=complex,
)

print(
    "Dipole matches expected matrix:",
    np.allclose(mu, mu_expected)
)

Dipole matches expected matrix: True


In [21]:
hermiticity_error_mu = norm(mu - mu.conj().T)

print("Dipole Hermiticity error =", hermiticity_error_mu)

Dipole Hermiticity error = 0.0


In [22]:
print("mu |gg> =")
print(mu @ gg)

print("\nmu_a |eg> + mu_b |ge> =")
print(mu_a * eg + mu_b * ge)

mu |gg> =
[0.+0.j 1.+0.j 1.+0.j 0.+0.j]

mu_a |eg> + mu_b |ge> =
[0.+0.j 1.+0.j 1.+0.j 0.+0.j]


In [23]:
mu_eig = eigenstates.conj().T @ mu @ eigenstates

print("Dipole operator in the energy eigenbasis:")
print(np.round(mu_eig, 6))

Dipole operator in the energy eigenbasis:
[[ 0.      +0.j -0.541196+0.j  1.306563+0.j  0.      +0.j]
 [-0.541196+0.j  0.      +0.j  0.      +0.j -0.541196+0.j]
 [ 1.306563+0.j  0.      +0.j  0.      +0.j  1.306563+0.j]
 [ 0.      +0.j -0.541196+0.j  1.306563+0.j  0.      +0.j]]


### Ground-to-single-excitation transition strengths

The intensity associated with an optical transition is proportional to
the squared magnitude of the dipole matrix element,

$$
I_{g\rightarrow\alpha}
\propto
|\langle \alpha|\mu|g\rangle|^2.
$$

We therefore calculate the transition strengths from the ground state
to the two coupled single-excitation eigenstates.

In [24]:
# Eigenstates are ordered by increasing energy
ket_ground = eigenstates[:, 0]
ket_minus = eigenstates[:, 1]
ket_plus = eigenstates[:, 2]
ket_double = eigenstates[:, 3]

# Transition amplitudes
mu_g_minus = np.vdot(ket_minus, mu @ ket_ground)
mu_g_plus = np.vdot(ket_plus, mu @ ket_ground)

# Transition strengths
strength_g_minus = abs(mu_g_minus)**2
strength_g_plus = abs(mu_g_plus)**2

print("Ground -> |-> transition amplitude =", mu_g_minus)
print("Ground -> |+> transition amplitude =", mu_g_plus)

print()
print("Ground -> |-> strength =", strength_g_minus)
print("Ground -> |+> strength =", strength_g_plus)

Ground -> |-> transition amplitude = (-0.5411961001461969+0j)
Ground -> |+> transition amplitude = (1.3065629648763766+0j)

Ground -> |-> strength = 0.29289321881345237
Ground -> |+> strength = 1.7071067811865477


In [25]:
total_strength = strength_g_minus + strength_g_plus
expected_total_strength = mu_a**2 + mu_b**2

print("Total single-excitation transition strength =", total_strength)
print("Expected =", expected_total_strength)

print(
    "Oscillator-strength sum correct:",
    np.allclose(total_strength, expected_total_strength)
)

Total single-excitation transition strength = 2.0
Expected = 2.0
Oscillator-strength sum correct: True


In [26]:
mu_minus_double = np.vdot(ket_double, mu @ ket_minus)
mu_plus_double = np.vdot(ket_double, mu @ ket_plus)

strength_minus_double = abs(mu_minus_double)**2
strength_plus_double = abs(mu_plus_double)**2

print("|-> -> |ee> strength =", strength_minus_double)
print("|+> -> |ee> strength =", strength_plus_double)

|-> -> |ee> strength = 0.29289321881345237
|+> -> |ee> strength = 1.7071067811865477


## 5. Local thermal Lindblad operators

Each two-level subsystem interacts with an independent thermal
environment.

For subsystem $a$,

$$
L_{a,\downarrow}
=
\sqrt{\gamma_a(\bar n_a+1)}\,\sigma_a^-,
$$

$$
L_{a,\uparrow}
=
\sqrt{\gamma_a\bar n_a}\,\sigma_a^+.
$$

Similarly, for subsystem $b$,

$$
L_{b,\downarrow}
=
\sqrt{\gamma_b(\bar n_b+1)}\,\sigma_b^-,
$$

$$
L_{b,\uparrow}
=
\sqrt{\gamma_b\bar n_b}\,\sigma_b^+.
$$

The downward operators remove an excitation, while the upward operators
describe excitation caused by the finite-temperature bath.

In [27]:
# Thermal Lindblad jump operators

L_a_down = np.sqrt(gamma_a * (nbar_a + 1.0)) * sigma_a_minus
L_a_up   = np.sqrt(gamma_a * nbar_a) * sigma_a_plus

L_b_down = np.sqrt(gamma_b * (nbar_b + 1.0)) * sigma_b_minus
L_b_up   = np.sqrt(gamma_b * nbar_b) * sigma_b_plus

In [28]:
down_prefactor_a = np.sqrt(gamma_a * (nbar_a + 1.0))
up_prefactor_a   = np.sqrt(gamma_a * nbar_a)

down_prefactor_b = np.sqrt(gamma_b * (nbar_b + 1.0))
up_prefactor_b   = np.sqrt(gamma_b * nbar_b)

print("Subsystem a:")
print("  downward prefactor =", down_prefactor_a)
print("  upward prefactor   =", up_prefactor_a)

print("\nSubsystem b:")
print("  downward prefactor =", down_prefactor_b)
print("  upward prefactor   =", up_prefactor_b)

Subsystem a:
  downward prefactor = 0.22912878474779202
  upward prefactor   = 0.05

Subsystem b:
  downward prefactor = 0.22912878474779202
  upward prefactor   = 0.05


In [29]:
print("a downward: |eg> -> proportional to |gg>")
print(L_a_down @ eg)

print("\na upward: |gg> -> proportional to |eg>")
print(L_a_up @ gg)

print("\nb downward: |ge> -> proportional to |gg>")
print(L_b_down @ ge)

print("\nb upward: |gg> -> proportional to |ge>")
print(L_b_up @ gg)

a downward: |eg> -> proportional to |gg>
[0.22912878+0.j 0.        +0.j 0.        +0.j 0.        +0.j]

a upward: |gg> -> proportional to |eg>
[0.  +0.j 0.05+0.j 0.  +0.j 0.  +0.j]

b downward: |ge> -> proportional to |gg>
[0.22912878+0.j 0.        +0.j 0.        +0.j 0.        +0.j]

b upward: |gg> -> proportional to |ge>
[0.  +0.j 0.  +0.j 0.05+0.j 0.  +0.j]


In [30]:
print(
    "a downward correct:",
    np.allclose(
        L_a_down @ eg,
        down_prefactor_a * gg
    )
)

print(
    "a upward correct:",
    np.allclose(
        L_a_up @ gg,
        up_prefactor_a * eg
    )
)

print(
    "b downward correct:",
    np.allclose(
        L_b_down @ ge,
        down_prefactor_b * gg
    )
)

print(
    "b upward correct:",
    np.allclose(
        L_b_up @ gg,
        up_prefactor_b * ge
    )
)

a downward correct: True
a upward correct: True
b downward correct: True
b upward correct: True


In [31]:
L_a_up_zeroT = np.sqrt(gamma_a * 0.0) * sigma_a_plus
L_b_up_zeroT = np.sqrt(gamma_b * 0.0) * sigma_b_plus

print("Zero-temperature upward norm a =", norm(L_a_up_zeroT))
print("Zero-temperature upward norm b =", norm(L_b_up_zeroT))

Zero-temperature upward norm a = 0.0
Zero-temperature upward norm b = 0.0


## 6. Liouvillian superoperator

The density matrix obeys the Lindblad master equation

$$
\frac{d\rho}{dt}
=
-\frac{i}{\hbar}[H_S,\rho]
+
\sum_k
\left(
L_k\rho L_k^\dagger
-
\frac{1}{2}
\{L_k^\dagger L_k,\rho\}
\right).
$$

Using column-stacking vectorization,

$$
\operatorname{vec}(A\rho B)
=
(B^T\otimes A)\operatorname{vec}(\rho),
$$

the master equation becomes

$$
\frac{d}{dt}|\rho\rangle\rangle
=
\mathcal{L}|\rho\rangle\rangle.
$$

For the Hamiltonian contribution,

$$
\mathcal{L}_H
=
-\frac{i}{\hbar}
\left(
I\otimes H_S
-
H_S^T\otimes I
\right).
$$

For a jump operator $L$,

$$
\mathcal{D}_L
=
L^*\otimes L
-
\frac{1}{2}
I\otimes L^\dagger L
-
\frac{1}{2}
(L^\dagger L)^T\otimes I.
$$

In [32]:
# Hilbert-space dimension
d = H_S.shape[0]

# Identity in the full two-TLS Hilbert space
I4 = np.eye(d, dtype=complex)


def vec(rho):
    """
    Column-stacking vectorization:
    vec(rho) = [column 1, column 2, ...].
    """
    return rho.reshape(-1, order="F")


def unvec(rho_vec):
    """
    Inverse of column-stacking vectorization.
    """
    return rho_vec.reshape((d, d), order="F")

In [33]:
rho_test = np.array(
    [
        [1.0, 2.0],
        [3.0, 4.0],
    ],
    dtype=complex,
)

print("rho_test =")
print(rho_test)

print("\nvec(rho_test) =")
print(rho_test.reshape(-1, order="F"))

rho_test =
[[1.+0.j 2.+0.j]
 [3.+0.j 4.+0.j]]

vec(rho_test) =
[1.+0.j 3.+0.j 2.+0.j 4.+0.j]


In [34]:
# Hamiltonian contribution to the Liouvillian

L_H = (
    -1j / hbar
    * (
        np.kron(I4, H_S)
        - np.kron(H_S.T, I4)
    )
)

print("Shape of L_H:", L_H.shape)

Shape of L_H: (16, 16)


In [35]:
def dissipator_super(L):
    """
    Liouville-space representation of

        D[L] rho
        = L rho L^\dagger
          - 1/2 {L^\dagger L, rho}

    using column-stacking vectorization.
    """

    LdagL = L.conj().T @ L

    jump_term = np.kron(L.conj(), L)

    left_term = np.kron(I4, LdagL)

    right_term = np.kron(LdagL.T, I4)

    return (
        jump_term
        - 0.5 * left_term
        - 0.5 * right_term
    )

In [36]:
D_a_down = dissipator_super(L_a_down)
D_a_up   = dissipator_super(L_a_up)

D_b_down = dissipator_super(L_b_down)
D_b_up   = dissipator_super(L_b_up)

print("D_a_down shape:", D_a_down.shape)
print("D_a_up shape:  ", D_a_up.shape)
print("D_b_down shape:", D_b_down.shape)
print("D_b_up shape:  ", D_b_up.shape)

D_a_down shape: (16, 16)
D_a_up shape:   (16, 16)
D_b_down shape: (16, 16)
D_b_up shape:   (16, 16)


In [37]:
L_super = (
    L_H
    + D_a_down
    + D_a_up
    + D_b_down
    + D_b_up
)

print("Full Liouvillian shape:", L_super.shape)

Full Liouvillian shape: (16, 16)


In [38]:
def dissipator_rho(L, rho):
    """
    Ordinary Hilbert-space Lindblad dissipator.
    """
    LdagL = L.conj().T @ L

    return (
        L @ rho @ L.conj().T
        - 0.5 * (LdagL @ rho + rho @ LdagL)
    )

In [39]:
def master_equation_direct(rho):

    hamiltonian_part = (
        -1j / hbar
        * (H_S @ rho - rho @ H_S)
    )

    dissipative_part = (
        dissipator_rho(L_a_down, rho)
        + dissipator_rho(L_a_up, rho)
        + dissipator_rho(L_b_down, rho)
        + dissipator_rho(L_b_up, rho)
    )

    return hamiltonian_part + dissipative_part

In [40]:
rho0 = np.outer(gg, gg.conj())

print("rho0 =")
print(rho0)

print("\nTrace rho0 =", np.trace(rho0))

rho0 =
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]]

Trace rho0 = (1+0j)


In [41]:
drho_direct = master_equation_direct(rho0)

In [42]:
drho_vec = L_super @ vec(rho0)
drho_super = unvec(drho_vec)

In [43]:
print(
    "Direct master equation matches Liouvillian:",
    np.allclose(drho_direct, drho_super)
)

print(
    "Difference norm =",
    norm(drho_direct - drho_super)
)

Direct master equation matches Liouvillian: True
Difference norm = 0.0


In [44]:
I_vec = vec(I4)

trace_bra = I_vec.conj().T

In [45]:
trace_preservation_vector = trace_bra @ L_super

trace_preservation_error = norm(trace_preservation_vector)

print(
    "Trace-preservation error =",
    trace_preservation_error
)

Trace-preservation error = 2.4532694666933987e-18


In [46]:
ground_state_derivative_norm = norm(
    L_super @ vec(rho0)
)

print(
    "||L rho0|| =",
    ground_state_derivative_norm
)

||L rho0|| = 0.006123724356957946


In [47]:
hermiticity_error_drho = norm(
    drho_super - drho_super.conj().T
)

print(
    "Hermiticity-preservation error =",
    hermiticity_error_drho
)

Hermiticity-preservation error = 0.0


## 7. Liouvillian eigenvalues and stationary state

The Liouvillian eigenmodes satisfy

$$
\mathcal{L}|r_\alpha\rangle\rangle
=
\lambda_\alpha |r_\alpha\rangle\rangle.
$$

For a stable open quantum system, the real parts of the Liouvillian
eigenvalues should be non-positive,

$$
\operatorname{Re}(\lambda_\alpha)\leq 0.
$$

A stationary state corresponds to a zero eigenvalue,

$$
\mathcal{L}|\rho_{\mathrm{ss}}\rangle\rangle=0.
$$

In [48]:
liouvillian_eigenvalues, liouvillian_eigenvectors = np.linalg.eig(L_super)

# Sort by magnitude so the eigenvalue closest to zero appears first
idx_L = np.argsort(np.abs(liouvillian_eigenvalues))

liouvillian_eigenvalues_sorted = liouvillian_eigenvalues[idx_L]

print("Liouvillian eigenvalues:")
for lam in liouvillian_eigenvalues_sorted:
    print(f"{lam.real:+.10f} {lam.imag:+.10f}j")

Liouvillian eigenvalues:
+0.0000000000 +0.0000000000j
-0.0550000000 +0.0000000000j
-0.0550000000 -0.0000000000j
-0.1100000000 +0.0000000000j
-0.0550000000 +0.2828427125j
-0.0550000000 -0.2828427125j
-0.0286784839 +0.9588030729j
-0.0286784839 -0.9588030729j
-0.0813215161 +0.9588030729j
-0.0813215161 -0.9588030729j
-0.0286784839 -1.2411969271j
-0.0286784839 +1.2411969271j
-0.0813215161 +1.2411969271j
-0.0813215161 -1.2411969271j
-0.0550000000 -2.2000000000j
-0.0550000000 +2.2000000000j


In [49]:
max_real_part = np.max(np.real(liouvillian_eigenvalues))

print("Largest real part =", max_real_part)

Largest real part = 4.0657581468206416e-19


In [50]:
tolerance = 1e-12

print(
    "All Liouvillian modes stable:",
    np.all(np.real(liouvillian_eigenvalues) <= tolerance)
)

All Liouvillian modes stable: True


In [51]:
zero_index = idx_L[0]

lambda_zero = liouvillian_eigenvalues[zero_index]

print("Eigenvalue closest to zero =", lambda_zero)

Eigenvalue closest to zero = (4.0657581468206416e-19+1.8535638170304925e-20j)


In [52]:
rho_ss_vec_raw = liouvillian_eigenvectors[:, zero_index]

In [53]:
rho_ss_raw = unvec(rho_ss_vec_raw)

print("Raw stationary-state matrix:")
print(rho_ss_raw)

print("\nRaw trace =", np.trace(rho_ss_raw))

Raw stationary-state matrix:
[[ 9.97737557e-01+0.00000000e+00j -1.17914877e-34-7.49133420e-35j
   7.55873992e-35-5.07519427e-36j  0.00000000e+00+0.00000000e+00j]
 [-3.27048652e-19+2.41420956e-19j  4.75113122e-02+1.10462807e-19j
  -6.99923102e-19+1.78147025e-18j -8.00701652e-35-2.79810583e-36j]
 [ 1.86086138e-18-8.64402777e-20j  4.40136846e-19-1.91280563e-18j
   4.75113122e-02+2.41798190e-19j -2.61127740e-34-1.49356783e-33j]
 [ 0.00000000e+00+0.00000000e+00j -1.45368556e-21-6.88725046e-21j
   2.21465953e-21+4.61031041e-20j  2.26244344e-03+3.07013587e-19j]]

Raw trace = (1.0950226244343892+6.592745838704925e-19j)


In [54]:
rho_ss = rho_ss_raw / np.trace(rho_ss_raw)

print("Normalized stationary state:")
print(np.round(rho_ss, 8))

print("\nTrace =", np.trace(rho_ss))

Normalized stationary state:
[[ 0.91115702-0.j -0.        -0.j  0.        -0.j  0.        +0.j]
 [-0.        +0.j  0.04338843+0.j -0.        +0.j -0.        -0.j]
 [ 0.        -0.j  0.        -0.j  0.04338843+0.j -0.        -0.j]
 [ 0.        +0.j -0.        -0.j  0.        +0.j  0.00206612+0.j]]

Trace = (1+9.62964972193618e-35j)


In [55]:
rho_ss_residual = norm(
    L_super @ vec(rho_ss)
)

print("Stationary-state residual =", rho_ss_residual)

Stationary-state residual = 2.088785819016661e-18


In [56]:
rho_ss_hermiticity_error = norm(
    rho_ss - rho_ss.conj().T
)

print(
    "Stationary-state Hermiticity error =",
    rho_ss_hermiticity_error
)

Stationary-state Hermiticity error = 3.1549248473516048e-18


In [57]:
rho_ss_eigenvalues = np.linalg.eigvalsh(
    0.5 * (rho_ss + rho_ss.conj().T)
)

print("Eigenvalues of rho_ss:")
print(rho_ss_eigenvalues)

print(
    "Minimum eigenvalue =",
    np.min(rho_ss_eigenvalues)
)

Eigenvalues of rho_ss:
[0.00206612 0.04338843 0.04338843 0.91115702]
Minimum eigenvalue = 0.0020661157024793385


In [58]:
zero_eigenvalues = np.sum(
    np.abs(liouvillian_eigenvalues) < 1e-10
)

print(
    "Number of zero Liouvillian eigenvalues =",
    zero_eigenvalues
)

Number of zero Liouvillian eigenvalues = 1


## 8. Light-matter interaction superoperator

The electric-dipole interaction is

$$
H_{\mathrm{int}}(t)=-\mu E(t).
$$

Its contribution to the density-matrix equation is

$$
\frac{d\rho}{dt}\bigg|_{\mathrm{int}}
=
\frac{i}{\hbar}E(t)[\mu,\rho].
$$

We therefore define the Liouville-space interaction superoperator

$$
V
=
\frac{i}{\hbar}
\left(
I\otimes\mu
-
\mu^T\otimes I
\right),
$$

so that

$$
V|\rho\rangle\rangle
=
\operatorname{vec}
\left(
\frac{i}{\hbar}[\mu,\rho]
\right).
$$

In [59]:
V = (
    1j / hbar
    * (
        np.kron(I4, mu)
        - np.kron(mu.T, I4)
    )
)

print("Shape of V:", V.shape)

Shape of V: (16, 16)


In [60]:
interaction_direct = (
    1j / hbar
    * (
        mu @ rho0
        - rho0 @ mu
    )
)

interaction_vec = V @ vec(rho0)
interaction_super = unvec(interaction_vec)

print(
    "Direct commutator matches V:",
    np.allclose(interaction_direct, interaction_super)
)

print(
    "Difference norm =",
    norm(interaction_direct - interaction_super)
)

Direct commutator matches V: True
Difference norm = 0.0


In [61]:
trace_preservation_V = trace_bra @ V

print(
    "Trace-preservation error for V =",
    norm(trace_preservation_V)
)

Trace-preservation error for V = 0.0


In [62]:
interaction_hermiticity_error = norm(
    interaction_super
    - interaction_super.conj().T
)

print(
    "Hermiticity error after applying V =",
    interaction_hermiticity_error
)

Hermiticity error after applying V = 0.0


## 9. Model-construction summary

In [63]:
print("========== MODEL SUMMARY ==========\n")

print("Hilbert-space dimension:   ", d)
print("Liouville-space dimension: ", d**2)

print("\n--- Hamiltonian and dipole ---")
print("H Hermiticity error:       ", norm(H_S - H_S.conj().T))
print("mu Hermiticity error:      ", norm(mu - mu.conj().T))
print("[H, N] error:              ", norm(H_S @ N - N @ H_S))

print("\n--- Liouvillian ---")
print("Liouvillian shape:         ", L_super.shape)
print("Trace-preservation error:  ", norm(trace_bra @ L_super))
print("Largest Re(lambda):        ", np.max(np.real(liouvillian_eigenvalues)))
print("Number of zero modes:      ", np.sum(np.abs(liouvillian_eigenvalues) < 1e-10))

print("\n--- Stationary state ---")
print("Steady-state residual:      ", norm(L_super @ vec(rho_ss)))
print("Trace rho_ss:              ", np.trace(rho_ss))
print("rho_ss Hermiticity error:  ", norm(rho_ss - rho_ss.conj().T))
print("Minimum rho_ss eigenvalue: ", np.min(rho_ss_eigenvalues))

print("\n--- Light-matter interaction ---")
print("V shape:                    ", V.shape)
print("V commutator check:         ", norm(interaction_direct - interaction_super))
print("V trace-preservation error: ", norm(trace_bra @ V))

========== MODEL SUMMARY ==========

Hilbert-space dimension:    4
Liouville-space dimension:  16

--- Hamiltonian and dipole ---
H Hermiticity error:        0.0
mu Hermiticity error:       0.0
[H, N] error:               0.0

--- Liouvillian ---
Liouvillian shape:          (16, 16)
Trace-preservation error:   2.4532694666933987e-18
Largest Re(lambda):         4.0657581468206416e-19
Number of zero modes:       1

--- Stationary state ---
Steady-state residual:       2.088785819016661e-18
Trace rho_ss:               (1+9.62964972193618e-35j)
rho_ss Hermiticity error:   3.1549248473516048e-18
Minimum rho_ss eigenvalue:  0.0020661157024793385

--- Light-matter interaction ---
V shape:                     (16, 16)
V commutator check:          0.0
V trace-preservation error:  0.0
